In [ ]:
import jax
jax.config.update("jax_enable_x64", True)


In [ ]:
import os
import time
from datetime import datetime

import numpy as np
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt

import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
tfb = tfp.bijectors

# --- new (scene) API ---------------------------------------------------------
from gigalens.jax.scene import Component, Plane, LensModel
from gigalens.jax.profiles.mass.epl import EPL
from gigalens.jax.profiles.light.sersic import SersicEllipse
from gigalens.jax.cosmo import w0waCDM_Cosmo
from gigalens.jax.scene_prob_model import ImageData, ProbModel
from gigalens.jax.scene_simulator import SceneSimulator
from gigalens.simulator import SimulatorConfig

# --- research-side pipeline / diagnostics (gigalens_research) ----------------
from gigalens_research.inference_utils import (
    InferenceContext, Pipeline, MAPStage, BridgeStage, MCLMCStage,
)
from gigalens_research.plotting import PosteriorReport, PipelineReport
import photutils.psf as psf

In [ ]:
import bullseye
import importlib
importlib.reload(bullseye)
from bullseye import make_5spl_bullseye, standard_pipeline, make_prob_grid

In [ ]:
#! THESE ARE DUPLICATED IN THE BULLSEYE FILE, NEED TO CHANGE BOTH
z_lens = 0.5 
z_source1 = 1.0
z_source2 = 1.5
z_source3 = 4.0
z_source4 = 7.0
z_source5 = 13.0

prob_model, prob_model_dr, truth_scene, truth_def_ratio = make_5spl_bullseye(sample_wa=False, snr_scale=4.)
ctx = InferenceContext.from_prob_model(prob_model)
ctx_dr = InferenceContext.from_prob_model(prob_model_dr)
pipeline = standard_pipeline(ctx)
pipeline_dr = standard_pipeline(ctx_dr)

In [ ]:
results_dir = os.path.join(
    os.path.expanduser("~"), "GIGALens-Code", "results",
    "sample_cosmology", "5spl_nfw_z13_highsnr_cosmology",
)
results_dir_dr = os.path.join(
    os.path.expanduser("~"), "GIGALens-Code", "results",
    "sample_cosmology", "5spl_nfw_z13_highsnr_dr_cosmology",
)
artifacts = pipeline.run(out_dir=results_dir, resume=True)
artifacts_dr = pipeline_dr.run(out_dir=results_dir_dr, resume=True)


In [ ]:
# pipeline_report = PipelineReport(pipeline)
# fig = pipeline_report.diagnostics("mclmc", chain=3)
# fig.show()

report = PosteriorReport(pipeline.posterior(), truth_x=truth_scene)
report.corner(kind="cosmology")
report.full_report()
plt.show()

In [ ]:
report_dr = PosteriorReport(pipeline_dr.posterior(), truth_x=truth_def_ratio)
report_dr.corner(kind=["geometry"])
report_dr.full_report()
plt.show()

In [ ]:
ps = pipeline_dr.posterior()

In [ ]:
idxes = [1,3,4,5]
dr_smps = jnp.stack([ps.flat_x[f'planes/{i}/geometry/deflection_ratio'] for i in idxes])
dr_median = jnp.median(dr_smps, axis=-1)
dr_cov = jnp.cov(dr_smps)
zs = [z_source1, z_source3, z_source4, z_source5]

In [ ]:
dr_median.shape

In [ ]:

import bullseye
importlib.reload(bullseye)
from bullseye import make_prob_grid
# observed = {
#     "deflect_ratio_obs": np.expand_dims(np.array(dr_medians), 1),
#     "deflect_ratio_obs_err": np.expand_dims(np.array(dr_stds), 1),
#     "z_source": np.expand_dims(np.array(zs), 1),
# }

cos=prob_model.model.cosmo.profile
observed = {
    "deflect_ratio_obs": np.expand_dims(dr_median, 1),
    "deflect_ratio_cov": dr_cov,
    # "deflect_ratio_cov_det": jnp.abs(jnp.linalg.det(dr_cov)),
    "z_source": np.expand_dims(np.array(zs), 1),
}
like, om_w_grid = make_prob_grid(observed, cos, n_grid=2000)

In [ ]:

# cosmo_prior = tfd.JointDistributionNamed(
#     dict(
#         Om0=tfd.Uniform(0.0, 1.0),
#         w0=tfd.Uniform(-2.0, -1/3),
#     )
# )

# # # chi2
# # def lens_cosmo_chi2(Om0, w0, z_source,
# #                     deflect_ratio_obs, deflect_ratio_cov_det, deflect_ratio_cov_inv,
# #                     cosmo):
# #     def mapped_dr(Om0, w0):
# #         return cosmo.deflection_ratio(z_source, Om0=Om0, w0=w0, H0=70., k=0.0)
# #     deflect_ratio = mapped_dr(Om0, w0)
# #     delta = deflect_ratio - deflect_ratio_obs
# #     chi2 = delta * deflect_ratio_cov_inv.dot(delta)
# #     return jnp.sum(chi2, axis=0)  # sum over redshifts

# # likelihood
# def lens_cosmo_loglike(Om0, w0, z_source,
#                     deflect_ratio_obs, deflect_ratio_cov_det, deflect_ratio_cov_inv,
#                     cosmo):
#     def mapped_dr(Om0, w0):
#         return cosmo.deflection_ratio(z_source, Om0=Om0, w0=w0, H0=70., k=0.0, wa=0.0)
#     deflect_ratio = mapped_dr(Om0, w0)
#     delta = deflect_ratio - deflect_ratio_obs
#     chi2 = delta * deflect_ratio_cov_inv.dot(delta)
#     normalization = jnp.log(2 * jnp.pi * deflect_ratio_cov_det)
#     log_like = -1 / 2 * (chi2 + normalization)
#     return jnp.sum(log_like, axis=0)  # sum over redshifts

# # log_prob
# def lens_cosmo_logprob(state, observation, cosmo):
#     log_prior = cosmo_prior.log_prob(state)
#     log_like = lens_cosmo_loglike(cosmo=cosmo, **state, **observation)
#     return log_like + log_prior


# def logsumexp_norm(log_prob):
#     c = jnp.max(log_prob)
#     logsumexp = c + jnp.log(jnp.nansum(jnp.exp(log_prob - c)))
#     return log_prob - logsumexp

    
# def make_prob_grid(observation, cosmo, n_grid=400, plane_indexes=None):
#     if plane_indexes is None:
#         plane_indexes = np.array(range(len(observation['z_source'])))
#         # print(plane_indexes)
    
#     Om0_grid, w0_grid = jnp.linspace(0, 1, n_grid), jnp.linspace(-2, -1/3, n_grid)
#     Om0_grid, w0_grid = jnp.meshgrid(Om0_grid, w0_grid, indexing='ij')
    
#     states = {
#         'Om0': Om0_grid.flatten(), 
#         'w0': w0_grid.flatten()
#     }

#     # obs_p = copy.deepcopy(observation)
#     selected_med = observation['deflect_ratio_obs'][plane_indexes]
#     selected_cov = observation['deflect_ratio_cov'][plane_indexes][:, plane_indexes]
#     # print(selected_cov)
#     obs_p = {}
#     obs_p["deflect_ratio_obs"] = selected_med
#     obs_p["deflect_ratio_cov_inv"] = jnp.linalg.inv(selected_cov)
#     obs_p["deflect_ratio_cov_det"] = jnp.abs(jnp.linalg.det(selected_cov))
#     obs_p["z_source"] = observation["z_source"][plane_indexes]

#     log_prob = lens_cosmo_logprob(states, obs_p, cosmo=cosmo)
#     log_prob = log_prob.reshape(Om0_grid.shape)
#     log_prob = np.array(log_prob).astype(np.float64)
#     log_prob = logsumexp_norm(log_prob)
#     prob = np.exp(log_prob)
#     return prob, (np.array(Om0_grid), np.array(w0_grid))




# # plotting posterior
# def get_ll_levels(like_grid, levels=[0.68, 0.955, 0.997]):
#     # Compute the density levels.
#     Hflat = like_grid.flatten()
#     inds = np.argsort(Hflat)[::-1]
#     Hflat = Hflat[inds]
#     sm = np.cumsum(Hflat)
#     sm /= sm[-1]
#     v = []
#     for level in levels:
#         v.append(Hflat[sm <= level][-1])
#     return sorted(v)

# # levels = get_ll_levels(like, levels=[0.68, 0.955, 0.997])
# # plt.contour(om_w_grid[0], om_w_grid[1], like, levels=levels, colors='k', linestyles=['--', ':', '-'])
# # plt.show()

In [ ]:
import corner
kw = lambda : dict(density=True)
stacked = np.stack([om_w_grid[0].flatten(),om_w_grid[1].flatten()]).T
fig = corner.corner(stacked,color="red", weights=np.array(like.flatten()), plot_datapoints=False, plot_density=False,truths=[0.3, -1.0], range=[(0.2, 0.4), (-1.2, -0.8)], hist_kwargs=kw(),)

cos_smps = np.stack([pipeline.posterior().flat_x[f'cosmo/Om0'], pipeline.posterior().flat_x[f'cosmo/w0']]).T
corner.corner(cos_smps, fig=fig, color='black', plot_datapoints=False, plot_density=False,hist_kwargs=kw(), labels=[r'$\Omega_m$', r'$w_0$'])
plt.show()
# om_w_grid[0].shape

In [ ]:
from matplotlib.patches import Patch

def plot_cosmo_contour(plane_indexes=None, fig=None, color='black', levels=(0.68, 0.95), label=None):
    like, om_w_grid = make_prob_grid(observed, cos, n_grid=1000, sample_wa=False, plane_indexes=plane_indexes)
    stacked = np.stack([om_w_grid[0].flatten(), om_w_grid[1].flatten()]).T
    fig = corner.corner(
        stacked,
        weights=np.array(like.flatten()),
        plot_datapoints=False,
        bins=50,
        contour_kwargs={'linewidths': 0.5},
        plot_density=False,
        fill_contours=True,
        truths=[0.3, -1.0],
        levels=levels,
        color=color,
        hist_kwargs=kw(),
        labels=[r'$\Omega_m$', r'$w_0$'],
        fig=fig,
    )

    if label is not None:
        if not hasattr(fig, '_legend_handles'):
            fig._legend_handles = []
        fig._legend_handles.append(Patch(facecolor=color, edgecolor=color, label=label))

        ndim = int(np.sqrt(len(fig.axes)))
        legend_ax = fig.axes[ndim - 1]  # top-right axis, standard corner-plot legend spot
        legend_ax.legend(handles=fig._legend_handles, loc='center', frameon=False)

    return fig

In [ ]:
fig=None
for i, c, l in zip([0, 1, 2, 3], ['y', 'g', 'blue', 'purple'], ['z=1, 1.5', 'z=1.5, 4', 'z=1.5, 7', 'z=1.5, 13']):
    
    fig =plot_cosmo_contour(plane_indexes=np.array([i,]), fig=fig, color=c, label = l)
fig = plot_cosmo_contour(fig=fig, label="All Planes")
fig.set_size_inches(10, 10)
fig.show()

In [ ]:

fig = plot_cosmo_contour(plane_indexes=np.array([0, 1,]), color='green', label="z=1, 1.5, 4")
plot_cosmo_contour(plane_indexes=np.array([1, 2,]), fig=fig, color='blue', label="z=1.5, 4, 7")
plot_cosmo_contour(plane_indexes=np.array([2, 3,]), fig=fig, color='purple', label="z=1.5, 7, 13")
plot_cosmo_contour(fig=fig, label="All Planes")
fig.set_size_inches(10, 10)
plt.show()

In [ ]:
fig = plot_cosmo_contour(plane_indexes=np.array([0, 1, 2]), color='green', label="z=1, 1.5, 4, 7")
plot_cosmo_contour(fig=fig, label="All Planes")
fig.set_size_inches(10, 10)

In [ ]:
dr_stds = jnp.std(dr_smps, axis=-1)


plt.subplot(211)
plt.errorbar(zs, dr_median,dr_stds,fmt='.', label='Measured deflection ratios')
plt.plot(jnp.linspace(0.5,15,101), cos.deflection_ratio(jnp.linspace(0.5,15,101), Om0=0.3, w0=-1, H0=70., wa=0.0, k=0.0),label=r'$\Lambda$CDM')
plt.title('Deflection ratio Hubble diagram')
plt.xlabel(r'Redshift')
plt.ylabel(r'$\eta$')
plt.legend()
plt.subplot(212)

dr_pred = cos.deflection_ratio(np.array(zs), Om0=0.3, w0=-1, H0=70., wa=0.0, k=0.0)
plt.errorbar(zs, dr_median-dr_pred,dr_stds,fmt='.', label='Measured difference')
plt.plot(jnp.linspace(0.5,15,101), jnp.zeros_like(jnp.linspace(0.5,15,101)),label=r'$\Lambda$CDM')
plt.title(r'Difference from $\Lambda$CDM')
plt.xlabel(r'Redshift')
plt.ylabel(r'$\Delta\eta$')

plt.tight_layout()
plt.show()